# Service Access | Silver Batch
**Water • Electricity • Internet | 2021–2025**

The current run processed **145 records**. All records mapped successfully, with no invalid records, duplicate keys, or reconciliation differences. The unchanged rerun made no changes.

## Run notes

- Run notebook cells in order.
- Allow one writer at a time.
- Before joining with other datasets in Gold, confirm that the shared region codes match.

## Validation status

The current batch and unchanged rerun passed. New-record, changed-value, and snapshot-removal behavior still need separate testing.


## Settings
Run ID is generated once per notebook run. Change coverage here; default mode is INCREMENTAL. FULL_SNAPSHOT requires declared complete slices later.

Valid out-of-scope years are reported and skipped. Missing measurements stay NULL. Invalid records block MERGE.


In [0]:
%sql
DECLARE OR REPLACE VARIABLE sa_processing_run_id STRING DEFAULT CONCAT('service_access_', UUID());


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_settings AS
SELECT 2021 AS start_year, 2025 AS end_year,
 session.sa_processing_run_id AS processing_run_id,
 'INCREMENTAL' AS refresh_mode;


In [0]:
%sql
SELECT ASSERT_TRUE(start_year >= 1 AND end_year >= start_year
 AND processing_run_id <> 'REPLACE_WITH_UNIQUE_RUN_ID'
 AND NULLIF(TRIM(processing_run_id),'') IS NOT NULL
 AND refresh_mode IN ('INCREMENTAL','FULL_SNAPSHOT'),
 'Configure year range, unique run ID and refresh mode in Settings.')
FROM service_access_settings;


## 1. Clean and select
Select drinking-water access, electricity access and internet access at home. Trim whitespace and cast year. The earlier 2021–2025 baseline had 145 selected rows, including missing values.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_clean_text AS
WITH cleaned AS (
 SELECT
  NULLIF(TRIM(REGEXP_REPLACE(country, r'\s+', ' ')), '') AS country,
  NULLIF(TRIM(year), '') AS year_raw,
  TRY_CAST(year AS INT) AS year,
  NULLIF(TRIM(REGEXP_REPLACE(region, r'\s+', ' ')), '') AS region_raw,
  NULLIF(TRIM(REGEXP_REPLACE(indicator, r'\s+', ' ')), '') AS service_measure,
  NULLIF(TRIM(REGEXP_REPLACE(breakdown_type, r'\s+', ' ')), '') AS breakdown_type,
  NULLIF(TRIM(REGEXP_REPLACE(breakdown_value, r'\s+', ' ')), '') AS breakdown_value,
  NULLIF(TRIM(value), '') AS value_raw,
  NULLIF(TRIM(REGEXP_REPLACE(unit, r'\s+', ' ')), '') AS unit,
  NULLIF(TRIM(REGEXP_REPLACE(source, r'\s+', ' ')), '') AS source,
  NULLIF(TRIM(notes), '') AS notes,
  _rescued_data,
  NULLIF(TRIM(_drive_file_id),'') AS source_file_id,
  NULLIF(TRIM(_source_file),'') AS source_file,
  NULLIF(TRIM(_run_id),'') AS ingestion_run_id, _ingested_at
 FROM pondolens.dev_bronze.service_access
)
SELECT *, CASE
 WHEN LOWER(service_measure) LIKE 'at least basic drinking-water%' THEN 'Water'
 WHEN service_measure = 'Electricity access' THEN 'Electricity'
 WHEN service_measure = 'Internet access at home' THEN 'Internet'
END AS service
FROM cleaned;

In [0]:
%sql
SELECT year, service, service_measure, unit, COUNT(*) AS total_rows
FROM service_access_clean_text WHERE service IS NOT NULL
GROUP BY year, service, service_measure, unit ORDER BY year, service;

### Percentage units
The five labels below match your successful run. Unknown labels block writing. Values must be 0–100; missing values remain NULL. Keep denominators separate.


In [0]:
%sql
SELECT DISTINCT unit FROM service_access_clean_text
WHERE service IS NOT NULL ORDER BY unit;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_approved_units AS
SELECT * FROM VALUES
 ('% of families'), ('% of households'), ('% of population'),
 ('% of rural population'), ('% of urban population') AS units(unit_label);
-- Observed labels in your successful dev run. New labels require source review.


## 2. Use the shared region crosswalk
All Silver notebooks must read the same verified mapping. Edit the table name and column aliases below to match your group's crosswalk.

**Required columns:** country, source_region, region_code, region_name, valid_from_year, valid_to_year.

The table name below is a proposed name, not a confirmed existing table. Do not create invented mappings to bypass this step.

- Map aliases such as `IV-A` and `REGION IVA` to the team's canonical CALABARZON code. Verify what `04` represents in each source first.
- Use one agreed national code for `PH` / `PHL`; keep national rows separate from regions.
- Validity years must reflect boundaries applicable to each observation. Do not map historical records to today's boundaries automatically.
- For source-dependent aliases, expose a source-specific adapter to the shared mapping; do not use an ambiguous country/label mapping.

One alias must match exactly one region for a given year. Changing canonical codes in an existing Silver table requires a coordinated migration because the code is part of the match key.


In [0]:
%sql
SELECT DISTINCT country, region_raw, year
FROM service_access_clean_text
WHERE service IS NOT NULL
ORDER BY country, region_raw, year;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_geography_reference AS
SELECT
 country,
 source_region,
 region_code,
 region_name,
 valid_from_year,
 valid_to_year
FROM pondolens.dev_silver.shared_region_crosswalk;
-- SETUP REQUIRED: replace the table name/column aliases with your verified shared crosswalk.
-- Use finite validity bounds; include a separate mapping for national observations.


In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Crosswalk has overlapping validity periods for the same alias. Fix mapping before continuing.')
FROM (
 SELECT *, MAX(valid_to_year) OVER (
  PARTITION BY LOWER(TRIM(country)), LOWER(TRIM(source_region))
  ORDER BY valid_from_year, valid_to_year, region_code
  ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
 ) AS previous_end_year
 FROM service_access_geography_reference
)
WHERE valid_from_year <= previous_end_year;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_expected AS
WITH selected AS (
 SELECT *, TRY_CAST(value_raw AS DECIMAL(18,6)) AS value
 FROM service_access_clean_text CROSS JOIN service_access_settings
 WHERE service IS NOT NULL AND year BETWEEN start_year AND end_year
)
SELECT s.*, NULLIF(TRIM(g.region_name),'') AS region,
 NULLIF(TRIM(g.region_code),'') AS region_code,
 EXISTS (SELECT 1 FROM service_access_approved_units u
  WHERE LOWER(s.unit) = u.unit_label) AS approved_percentage_unit
FROM selected s LEFT JOIN service_access_geography_reference g
 ON LOWER(TRIM(s.country)) = LOWER(TRIM(g.country))
 AND LOWER(TRIM(s.region_raw)) = LOWER(TRIM(g.source_region))
 AND s.year BETWEEN g.valid_from_year AND g.valid_to_year;


## 3. Check before writing
Zero invalid records, duplicate keys and invalid crosswalk rows. `out_of_scope_rows` is informational and does not block a nonempty valid batch. Selected and mapped counts must match. Missing observations are allowed. If a check fails, inspect the records; keep this validation enabled.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_batch_gate AS
SELECT
 (SELECT COUNT(*) FROM service_access_expected) AS expected_rows,
 (SELECT COUNT(*) FROM service_access_clean_text CROSS JOIN service_access_settings WHERE service IS NOT NULL
   AND year BETWEEN start_year AND end_year) AS selected_rows,
 (SELECT COUNT(*) FROM service_access_clean_text WHERE service IS NOT NULL
   AND (year IS NULL OR year < 1)) AS invalid_years,
 (SELECT COUNT(*) FROM service_access_clean_text CROSS JOIN service_access_settings
  WHERE service IS NOT NULL AND year >= 1
  AND year NOT BETWEEN start_year AND end_year) AS out_of_scope_rows,
 (SELECT COUNT(*) FROM service_access_expected
  WHERE country IS NULL OR LOWER(country) <> 'philippines' OR region_code IS NULL OR region IS NULL
   OR source IS NULL OR unit IS NULL
   OR source_file_id IS NULL OR ingestion_run_id IS NULL
   OR NULLIF(TRIM(_rescued_data),'') IS NOT NULL
   OR (value_raw IS NOT NULL AND value IS NULL)
   OR (NOT approved_percentage_unit OR value < 0 OR value > 100)) AS invalid_rows,
 (SELECT COUNT(*) FROM (
  SELECT country, year, region_code, service_measure, breakdown_type,
   breakdown_value, unit, source FROM service_access_expected
  GROUP BY country, year, region_code, service_measure, breakdown_type,
   breakdown_value, unit, source HAVING COUNT(*) > 1)) AS duplicate_keys,
 (SELECT COUNT(*) FROM service_access_geography_reference
 WHERE NULLIF(TRIM(country),'') IS NULL OR NULLIF(TRIM(source_region),'') IS NULL
 OR NULLIF(TRIM(region_code),'') IS NULL OR NULLIF(TRIM(region_name),'') IS NULL
 OR valid_from_year IS NULL OR valid_to_year IS NULL
 OR valid_from_year > valid_to_year) AS invalid_crosswalk;


In [0]:
%sql
SELECT * FROM service_access_batch_gate;

In [0]:
%sql
SELECT ASSERT_TRUE(
 expected_rows > 0 AND expected_rows = selected_rows
 AND invalid_years = 0 AND invalid_rows = 0
 AND duplicate_keys = 0 AND invalid_crosswalk = 0,
 CONCAT('Batch blocked. Invalid records=', invalid_rows,
 ', duplicate keys=', duplicate_keys, ', invalid years=', invalid_years,
 ', invalid crosswalk=', invalid_crosswalk,
 ', mapped rows=', expected_rows, ', selected rows=', selected_rows)
)
FROM service_access_batch_gate;

In [0]:
%sql
SELECT year, region_raw, service_measure, value_raw, unit, source
FROM service_access_expected
WHERE region IS NULL OR region_code IS NULL
 OR source IS NULL OR unit IS NULL
 OR (value_raw IS NOT NULL AND value IS NULL)
 OR (NOT approved_percentage_unit OR value < 0 OR value > 100);

## 4. Check schema and create the target
The schema gate rejects an incompatible existing table before MERGE. It does not convert STRING hash IDs to INT.

For an old schema: stop the batch, back up the table, build a replacement with an old-to-new ID map, validate it, and coordinate downstream references before switching. Do not drop the old table to bypass validation.

Traceability fields identify the source file, ingestion run and processing run. `processed_at` uses UTC.


In [0]:
%sql
SET TIME ZONE 'UTC';

In [0]:
%sql
WITH required AS (
 SELECT * FROM VALUES
 ('service_access_id','INT'),
 ('year','INT'),
 ('unit','STRING'),
 ('service','STRING'),
 ('service_measure','STRING'),
 ('region','STRING'),
 ('region_code','STRING'),
 ('breakdown_type','STRING'),
 ('breakdown_value','STRING'),
 ('value','DECIMAL(18,6)'),
 ('source','STRING'),
 ('source_file_id','STRING'),
 ('source_file','STRING'),
 ('ingestion_run_id','STRING'),
 ('processing_run_id','STRING'),
 ('processed_at','TIMESTAMP')
 AS schema_contract(column_name, type_name)
), existing AS (
 SELECT LOWER(column_name) AS column_name, UPPER(full_data_type) AS type_name
 FROM pondolens.information_schema.columns
 WHERE table_schema = 'dev_silver' AND table_name = 'service_access'
), differences AS (
 SELECT r.column_name FROM required r LEFT JOIN existing e
 ON r.column_name = e.column_name
 WHERE e.column_name IS NULL OR r.type_name <> e.type_name
)
SELECT ASSERT_TRUE(
 (SELECT COUNT(*) FROM existing) = 0 OR
 ((SELECT COUNT(*) FROM differences) = 0
  AND (SELECT COUNT(*) FROM existing) = (SELECT COUNT(*) FROM required)),
 'Existing Silver schema differs. Use a controlled migration before running this notebook.') AS validation;
-- Requires visibility of the target metadata in Unity Catalog.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.service_access (
 service_access_id INT, year INT, unit STRING, service STRING,
 service_measure STRING, region STRING, region_code STRING,
 breakdown_type STRING, breakdown_value STRING,
 value DECIMAL(18,6), source STRING, source_file_id STRING, source_file STRING,
 ingestion_run_id STRING, processing_run_id STRING, processed_at TIMESTAMP
) USING DELTA;


## 5. Check existing Silver keys
IDs and observation keys must already be unique before MERGE.

In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT service_access_id)
 AND COUNT_IF(service_access_id IS NULL OR service_access_id <= 400000000) = 0
 AND COUNT_IF(year IS NULL OR region_code IS NULL OR service_measure IS NULL OR unit IS NULL OR source IS NULL) = 0,
 'Target has invalid or duplicate IDs/required keys')
FROM pondolens.dev_silver.service_access;

In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,'Target has duplicate business keys') FROM (
 SELECT year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 FROM pondolens.dev_silver.service_access
 GROUP BY year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 HAVING COUNT(*) > 1);

### Missing-source-key check
INCREMENTAL mode cannot identify deletions: omitted rows are retained.

For FULL_SNAPSHOT mode, declare every complete `(year, source, service_measure)` slice below—including an empty slice when its previous records were removed. Scope must match Settings. Missing keys are displayed and block MERGE for review; no records are automatically deleted.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_refresh_scope AS
SELECT CAST(NULL AS INT) AS year,
 CAST(NULL AS STRING) AS source,
 CAST(NULL AS STRING) AS service_measure
WHERE FALSE;
-- FULL_SNAPSHOT: replace this SELECT with the verified complete slices.
-- Example structure: SELECT * FROM VALUES
-- (2024, '<exact source>', 'Electricity access') AS scope(year, source, service_measure);


In [0]:
%sql
SELECT ASSERT_TRUE(refresh_mode = 'INCREMENTAL' OR (
 (SELECT COUNT(*) FROM service_access_refresh_scope) > 0 AND
 (SELECT COUNT(*) FROM service_access_refresh_scope
  WHERE year IS NULL OR year NOT BETWEEN start_year AND end_year
   OR NULLIF(TRIM(source),'') IS NULL OR NOT (LOWER(service_measure) LIKE 'at least basic drinking-water%'
    OR service_measure IN ('Electricity access','Internet access at home'))
   OR service_measure IS NULL) = 0 AND
 (SELECT COUNT(*) FROM service_access_expected s WHERE NOT EXISTS (
  SELECT 1 FROM service_access_refresh_scope r
  WHERE s.year = r.year AND s.source = r.source AND s.service_measure = r.service_measure)) = 0),
 'FULL_SNAPSHOT requires explicit complete slices matching the incoming batch and year range.')
FROM service_access_settings;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_missing_source_keys AS
SELECT t.* FROM pondolens.dev_silver.service_access t
CROSS JOIN service_access_settings cfg
WHERE cfg.refresh_mode = 'FULL_SNAPSHOT'
 AND EXISTS (SELECT 1 FROM service_access_refresh_scope r
  WHERE t.year = r.year AND t.source = r.source AND t.service_measure = r.service_measure)
 AND NOT EXISTS (SELECT 1 FROM service_access_expected s WHERE s.year <=> t.year AND s.region_code <=> t.region_code AND s.service_measure <=> t.service_measure AND s.breakdown_type <=> t.breakdown_type AND s.breakdown_value <=> t.breakdown_value AND s.unit <=> t.unit AND s.source <=> t.source);


In [0]:
%sql
SELECT * FROM service_access_missing_source_keys;


In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Snapshot has missing source keys. Review removal/correction before MERGE; no automatic deletion.')
FROM service_access_missing_source_keys;


## 6. Prepare batch IDs
Existing observations retain IDs. New observations receive IDs above the current maximum.
The match key is year + region_code + service_measure + breakdown + unit + source. `<=>` compares nullable fields safely. No automatic deletion or deduplication.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_batch_upsert AS
WITH input AS (
 SELECT s.* FROM service_access_expected s CROSS JOIN service_access_batch_gate g
 WHERE CASE WHEN g.expected_rows > 0 AND g.expected_rows = g.selected_rows
 AND g.invalid_years = 0 AND g.invalid_rows = 0 AND g.duplicate_keys = 0 AND g.invalid_crosswalk = 0
 THEN TRUE ELSE CAST(RAISE_ERROR('Batch validation failed') AS BOOLEAN) END
), matched AS (
 SELECT s.*, t.service_access_id AS existing_id
 FROM input s LEFT JOIN pondolens.dev_silver.service_access t ON s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source
), new_rows AS (
 SELECT *, ROW_NUMBER() OVER (ORDER BY year, region_code, service_measure,
 breakdown_type, breakdown_value, unit, source) AS new_sequence
 FROM matched WHERE existing_id IS NULL
), max_id AS (
 SELECT COALESCE(MAX(service_access_id),400000000) AS last_id
 FROM pondolens.dev_silver.service_access
)
SELECT existing_id AS service_access_id, year, unit, service, service_measure,
 region, region_code, breakdown_type, breakdown_value, value, source,
 source_file_id, source_file, ingestion_run_id, cfg.processing_run_id
FROM matched CROSS JOIN service_access_settings cfg WHERE existing_id IS NOT NULL
UNION ALL
SELECT CASE WHEN CAST(m.last_id AS BIGINT) + s.new_sequence <= 2147483647
 THEN CAST(CAST(m.last_id AS BIGINT) + s.new_sequence AS INT)
 ELSE CAST(RAISE_ERROR('INT ID allocation exhausted') AS INT) END,
 s.year, s.unit, s.service, s.service_measure, s.region, s.region_code,
 s.breakdown_type, s.breakdown_value, s.value, s.source,
 s.source_file_id, s.source_file, s.ingestion_run_id, cfg.processing_run_id
FROM new_rows s CROSS JOIN max_id m CROSS JOIN service_access_settings cfg;

## 7. MERGE
Insert new observations and update changed measurements or labels. Unchanged observations keep IDs, lineage and processed_at—even if ingested again.

Lineage describes the last inserted/changed observation. A rerun does not rewrite every record merely to stamp the new run. No automatic deletions.


### Capture the planned changes
The signature records IDs, processing timestamps and lineage before MERGE. If no changes are planned, the post-write signature must be identical. This checks unchanged reruns without writing snapshots to a shared table.


In [0]:
%sql
DECLARE OR REPLACE VARIABLE sa_before_signature STRING;
DECLARE OR REPLACE VARIABLE sa_planned_changes BIGINT;


In [0]:
%sql
SET VAR sa_before_signature = (
 SELECT SHA2(TO_JSON(SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
  'id', service_access_id, 'processed_at', processed_at,
  'source_file_id', source_file_id, 'source_file', source_file,
  'ingestion_run_id', ingestion_run_id, 'processing_run_id', processing_run_id
 )))),256)
 FROM pondolens.dev_silver.service_access
);


In [0]:
%sql
SET VAR sa_planned_changes = (
 SELECT COUNT(*) FROM service_access_batch_upsert s
 LEFT JOIN pondolens.dev_silver.service_access t
 ON s.year <=> t.year AND s.region_code <=> t.region_code
 AND s.service_measure <=> t.service_measure
 AND s.breakdown_type <=> t.breakdown_type
 AND s.breakdown_value <=> t.breakdown_value
 AND s.unit <=> t.unit AND s.source <=> t.source
 WHERE t.service_access_id IS NULL OR NOT(t.value <=> s.value)
 OR NOT(t.region <=> s.region) OR NOT(t.service <=> s.service)
);


In [0]:
%sql
MERGE INTO pondolens.dev_silver.service_access t
USING service_access_batch_upsert s
ON s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source
WHEN MATCHED AND (
 NOT (t.value <=> s.value) OR NOT (t.region <=> s.region)
 OR NOT (t.service <=> s.service)
) THEN UPDATE SET
 t.value = s.value, t.region = s.region, t.service = s.service,
 t.source_file_id = s.source_file_id, t.source_file = s.source_file,
 t.ingestion_run_id = s.ingestion_run_id,
 t.processing_run_id = s.processing_run_id,
 t.processed_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
 service_access_id, year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source,
 source_file_id, source_file, ingestion_run_id, processing_run_id, processed_at
) VALUES (
 s.service_access_id, s.year, s.unit, s.service, s.service_measure,
 s.region, s.region_code, s.breakdown_type, s.breakdown_value,
 s.value, s.source, s.source_file_id, s.source_file, s.ingestion_run_id,
 s.processing_run_id, CURRENT_TIMESTAMP()
);

## 8. Check results
Counts must MATCH; the record comparison should return zero rows. Reconciliation checks the incoming batch; earlier observations outside the batch are retained.

In [0]:
%sql
WITH counts AS (
 SELECT (SELECT COUNT(*) FROM service_access_expected) AS batch_rows,
 (SELECT COUNT(*) FROM pondolens.dev_silver.service_access t
 WHERE EXISTS (SELECT 1 FROM service_access_expected s WHERE s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source)) AS matched_silver_rows
)
SELECT *, ASSERT_TRUE(batch_rows = matched_silver_rows,
 'Reconciliation failed: incoming and matched Silver counts differ.') AS validation
FROM counts;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_record_differences AS
WITH expected AS (
 SELECT year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source FROM service_access_expected
), actual AS (
 SELECT year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source FROM pondolens.dev_silver.service_access t
 WHERE EXISTS (SELECT 1 FROM service_access_expected s WHERE s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source)
), missing AS (SELECT * FROM expected EXCEPT ALL SELECT * FROM actual),
extra AS (SELECT * FROM actual EXCEPT ALL SELECT * FROM expected)
SELECT 'MISSING_OR_CHANGED' AS issue, * FROM missing
UNION ALL SELECT 'EXTRA_OR_CHANGED' AS issue, * FROM extra;

In [0]:
%sql
SELECT * FROM service_access_record_differences;


In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Reconciliation failed: records differ between cleaned batch and Silver.')
FROM service_access_record_differences;


In [0]:
%sql
SELECT ASSERT_TRUE(
 COUNT(*) = COUNT(DISTINCT service_access_id)
 AND SUM(CASE WHEN service_access_id IS NULL OR service_access_id < 400000001 THEN 1 ELSE 0 END) = 0
 AND COUNT_IF(source_file_id IS NULL OR ingestion_run_id IS NULL
  OR processing_run_id IS NULL OR processed_at IS NULL) = 0,
 'Post-write IDs or lineage failed. Repair existing records before Gold loading.')
FROM pondolens.dev_silver.service_access;


In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,'Post-write duplicate observation keys.') FROM (
 SELECT year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 FROM pondolens.dev_silver.service_access
 GROUP BY year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 HAVING COUNT(*) > 1
);


In [0]:
%sql
SELECT session.sa_planned_changes AS planned_changes,
 ASSERT_TRUE(session.sa_planned_changes > 0 OR
 session.sa_before_signature <=> SHA2(TO_JSON(SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
  'id', service_access_id, 'processed_at', processed_at,
  'source_file_id', source_file_id, 'source_file', source_file,
  'ingestion_run_id', ingestion_run_id, 'processing_run_id', processing_run_id
 )))),256),
 'No-change batch altered IDs, timestamps or lineage.') AS validation
FROM pondolens.dev_silver.service_access;


In [0]:
%sql
SELECT service, year, COUNT(*) AS rows,
 COUNT(value) AS observed_values,
 COUNT(*) - COUNT(value) AS missing_values
FROM pondolens.dev_silver.service_access
GROUP BY service, year ORDER BY service, year;

## 9. Verify rerun
Rerun unchanged input: no inserted/updated records, and existing IDs/processed_at stay unchanged. Inspect MERGE operation metrics below. If metrics are absent, compare before/after snapshots.
**BP/AP3:** regional access can provide needs context when geography, years, units and source definitions are compatible. National electricity observations must not be repeated as regional data. Do not sum percentages. AP1/AP2 need other spending/execution tables.


In [0]:
%sql
DESCRIBE HISTORY pondolens.dev_silver.service_access LIMIT 3;

## Release checks
1. Run this candidate twice with unchanged input. Second run: planned_changes = 0, signature passes, MERGE inserts/updates/deletes = 0.
2. In isolated test tables, verify a new row, a changed value with unchanged ID, rejection of invalid percentage/duplicate key/unmapped region, and a missing key in FULL_SNAPSHOT mode. No destructive production tests.
3. Compare the canonical codes below with the other Silver tables. Compatible boundaries must also be confirmed; successful mapping alone does not establish comparability.
4. Schedule Bronze ingestion → this Silver notebook → Gold. Use a service principal with Bronze/crosswalk read and target write permissions, maximum concurrent runs 1, failure notifications and run history.
5. Keep the deployed schema/table names configured consistently. Do not recreate the shared crosswalk or auto-migrate an incompatible target.

A failed post-write assertion leaves the Delta MERGE committed. Stop downstream Gold tasks until reviewed; notebook cells are not one multi-statement transaction.

**BP/AP3:** use recorded regional values with compatible years, source definitions and denominators. National observations stay separate. Do not sum percentages. 2024 access used for 2025 spending must retain its 2024 label.


In [0]:
%sql
SELECT DISTINCT region_code, region FROM pondolens.dev_silver.service_access
ORDER BY region_code;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_region_fix AS
SELECT * FROM VALUES
 ('01', 'I'),    ('02', 'II'),   ('03', 'III'),
 ('04', 'IVA'),  ('05', 'V'),    ('06', 'VI'),
 ('07', 'VII'),  ('08', 'VIII'), ('09', 'IX'),
 ('10', 'X'),    ('11', 'XI'),   ('12', 'XII'),
 ('13', 'NCR'),  ('14', 'CAR'),  ('15', 'BARMM'),
 ('16', 'XIII'), ('17', 'IVB'),  ('PHL', 'PHL')
AS mapping(old_code, new_code);

In [0]:
%sql
SELECT DISTINCT
 s.region,
 s.region_code AS current_code,
 m.new_code
FROM pondolens.dev_silver.service_access s
LEFT JOIN service_access_region_fix m
 ON s.region_code = m.old_code
ORDER BY current_code;

In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Conversion would create duplicate observation keys.')
FROM (
 SELECT
  year,
  COALESCE(m.new_code, s.region_code) AS final_code,
  service_measure, breakdown_type, breakdown_value, unit, source
 FROM pondolens.dev_silver.service_access s
 LEFT JOIN service_access_region_fix m
  ON s.region_code = m.old_code
 GROUP BY
  year, COALESCE(m.new_code, s.region_code),
  service_measure, breakdown_type, breakdown_value, unit, source
 HAVING COUNT(*) > 1
);

In [0]:
%sql
MERGE INTO pondolens.dev_silver.service_access t
USING service_access_region_fix m
ON t.region_code = m.old_code
WHEN MATCHED AND t.region_code <> m.new_code
THEN UPDATE SET
 t.region_code = m.new_code,
 t.processing_run_id = 'service_access_region_code_migration_01',
 t.processed_at = CURRENT_TIMESTAMP();

In [0]:
%sql
MERGE INTO pondolens.dev_silver.shared_region_crosswalk t
USING service_access_region_fix m
ON t.region_code = m.old_code
WHEN MATCHED AND t.region_code <> m.new_code
THEN UPDATE SET t.region_code = m.new_code;

In [0]:
%sql
SELECT region_code, COUNT(*) AS row_count
FROM pondolens.dev_silver.service_access
GROUP BY region_code
ORDER BY region_code;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT region_code) AS distinct_codes
FROM pondolens.dev_silver.service_access;

In [0]:
%sql
SELECT region_code, COUNT(*) AS row_count
FROM pondolens.dev_silver.service_access
GROUP BY region_code
ORDER BY region_code;

In [0]:
%sql
SELECT region_code, COUNT(*) AS row_count
FROM pondolens.dev_silver.service_access
WHERE region_code <> 'PHL'
GROUP BY region_code
ORDER BY region_code;

In [0]:
%sql
SELECT
  region AS region_raw,
  year,
  COUNT(*) AS row_count
FROM pondolens.dev_bronze.service_access
WHERE LOWER(region) LIKE '%negros%'
   OR UPPER(region) LIKE '%NIR%'
GROUP BY region, year
ORDER BY year, region;

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.service_access;

In [0]:
%sql
SELECT DISTINCT region
FROM pondolens.dev_bronze.service_access
ORDER BY region;